## AI4Climate ML tutorial - Training in PyTorch
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we will use the previously prepared tabular dataset to predict climate zones. We will train on different eras to see how well the results generalise with climate change. 

### Prerequisites 
- Same as previous  notebooks
- Have completed training pipeline, inference and evaluation notebooks.


### Learning outcomes from completing the notebook

- Understand how to build a pipeline using pytorch
- Understand the core pytorch concepts and classes
- Understand bho to manage experiments using ML Flow

## Tutorial - Creating a machine learning pipe

Further Reading
* [PyTorch Docs](https://pytorch.org/)

## Setup
First we start by loading the data we have prepared previously, and other set up elements

### Environment 
This notebook uses the environment defined in this repository in the [requirements.txt](../requirements.txt) file (venv) or [requirements.yaml](../requirements.yaml).  

### Imports

In [1]:
import pathlib
import os
import datetime
import json

In [2]:
import numpy 
import pandas

In [3]:
import matplotlib
import matplotlib.pyplot

In [4]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [5]:
import mlflow

In [6]:
import torch

## Load and prepare data 
We will now load the dataset and do the usual data prep steps, like train/test split and normalisation.


#### Dataset parameters

In [7]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, ver

In [8]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [9]:
current_platform = tutorial_config['platform']

In [10]:
current_platform

'jasmin'

In [11]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones')

In [12]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones/ml_ready')

In [13]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [14]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

#### Load data for training

In [15]:
current_res = 1.0

In [16]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones/ml_ready/climate_zones_1p0.csv')

In [17]:
zones_df = pandas.read_csv(mlready_data_path)

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

#### Selecting features

In [18]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [19]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [20]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

#### Train/test split

In [21]:
random_seed = tutorial_config['random_seed']

In [22]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [23]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [24]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


## Using Pytorch

From this point in the tutorial, we diverge from what was done in the ml training pipeline tutorial as instead of setting up and training our model in scikit-learn, we're going to use a more sophisticated machine learning library called pytorch. This gives us more more control over how we implement our neural network and gives us much more power to use advanced architectures, losss functions and distributed computing techniques.

When using pytorch, the main difference is that we will specify the details of the nerual netowrk architecture and training loop much more explictly. This means we can customise and optimise these details for the particular problem. Key elements of a pytorch training pipeline, compared to what was previous described are as follows:
1. Data Loading and Cleaning - This is usually done through a pytorch Dataset class. In addition one creates a Dataset Loader object, which whas the responsibility for iterating through the data in the dataset, including shuffling data between epoch where appropriate.
2. Feature Engineering - Same as before, but may be a part of the dataset class
3. Train/test split - Same method as before. Usually different dataset objects will represent the train, validate and test sets.
4. Data Preparation - Same as before, but code may be structured differently to have the normalisation and scaling happening inside the pytorch dataset class.
5. Algorithm Setup - Usuall one creates a class representing the model architecture. One then also speciied key hyperparameters such as the optimiser for training the weights, the learning rate and similar. 
6. Algorithm Training - The elements of the training loop are described more explictly in pytorch typically with a explicit loop for iterating through batches and an outer loop for iterating through batches.
7. Inference - The model object is used as a callable object for producing 
8. Evaluation - Evaluation is the same as before. 
9. Interpretability and Explainability - As model architectures become more complex, it becomes more difficult to explain or interpret the results, this is an active area of research.
10. Model storage - Pytorch has a more sophisticated mechanism for [saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).


### Check GPU availability

As a part of this notebook, we will now be training our models using the GPUs on JASMIN. There are some additional steps required for this purpose, such as moving the data and model onto the GPU memory for processing. ML frameworks are especially helpful for this in abtracting away many of the details of this into a few commands.

In this cell, we check whether there is a GPU to use. [CUDA](https://en.wikipedia.org/wiki/CUDA) is the underlying software layer that interfaces to nvidia GPUs. This check allows the notebook to seamlessly work either ion a gpu if one is available or do processing on a cpu when a gpu is not avaiulable.



In [25]:
# Autodetect GPU and use if possible
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device

device(type='cuda', index=0)

#### Training hyperparameters

In [26]:
training_params = {
    'batch_size': 16,
    'num_epochs': 10,
    'learning_rate': 0.001,
    'loss': 'CrossEntropyLoss',
    'criterion': 'CrossEntropyLoss',
    'optimizer': 'Adam',
}

### Define a data loader

Pytorch defines the interface between the dataset and machine learning through a base class (`torch.utils.data.Dataset), which follow [pythonic paradigms](https://realpython.com/ref/glossary/pythonic/) for software architecture. The implementation of the class hides away the details of the data being used so that the dataset can be used in the generic pytorch architecture. There is also then a data loader is an [iterator](https://www.w3schools.com/python/python_iterators.asp) on the dataset, enabling pytorch to progress through all the data during training. Ultimately the aim of this data architecture is to present the data in the correct [pytorch tensor format](https://docs.pytorch.org/docs/stable/tensors.html) expected by the neural network for training purposes.

Key Concepts:
- **Pytorch Dataset**: Handles loading and preparing the data for use with pytorch. Implements key [python built-in methods](), including:
  - [`__init__`](https://docs.python.org/3/reference/datamodel.html#object.__init__) creates the dataset object, and typically loads and transforms the data ready for use, or in a lazy loading paradigm, define the task pipeline for loading the data upon request.
  - [`__len__`](https://docs.python.org/3/reference/datamodel.html#object.__len__) specifies how many data point there are
  - [`__getitem__`](https://docs.python.org/3/reference/datamodel.html#object.__getitem__) return the item for a particular index
- [**Pytorch Data Loader**](https://docs.pytorch.org/docs/stable/data.html): Interfaces between the ML algorithm being trained and the dataset, selecting mini batches of data to use with each iteration of the gradient descent with back propogation used for training the neural network. Key parameters include:
  - *dataset* - The dataset to load the data from (as described above).
  - [*batch size*](https://www.geeksforgeeks.org/deep-learning/batch-size-in-neural-network/) - How many samples to use in each mini batch during training.
  - [*shuffle*](https://stats.stackexchange.com/questions/245502/why-should-we-shuffle-data-while-training-a-neural-network) - Whether to shuffle the order of data use during each of the epochs. Data shuffling improves the generalisation of what is learned by the neural network.


Further reading
- [Intro to datasets and data loader - pytorch docs](https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html)
- [Tutorial on using CSV data with pytorch data architecture - Machine Learning Mastery](https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/)
- [Converting pnadas dataframe to the pytorch](https://www.geeksforgeeks.org/deep-learning/converting-a-pandas-dataframe-to-a-pytorch-tensor/)
- [Encoding target data for a neural network using LabelBinarizer - scikit learn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html)

### Implementation details for this tutorial

In [27]:
class ClimateZonesDataset(torch.utils.data.Dataset):
    """
    Inspired by this tutorial:
    https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
    """
    def __init__(self, df_ml, predictor_features, target_feature, device, stats_dict=None):
        self._df_ml = df_ml.reset_index().drop(['index'],axis='columns')
        self._device = device
        self.input_scaler = sklearn.preprocessing.StandardScaler()
        if stats_dict is None:
            self.input_scaler.fit(self._df_ml[predictor_features])
        else:
            self.input_scaler.mean_ = numpy.array(stats_dict['input_mean'])
            self.input_scaler.scale_ = numpy.array(stats_dict['input_scale'])

        self._X = torch.tensor(self.input_scaler.transform(self._df_ml[predictor_features]),  
                               dtype=torch.float32)


        self.target_encoder = sklearn.preprocessing.LabelBinarizer(sparse_output=False)
        if stats_dict is None:
            self.target_encoder.fit(self._df_ml[[target_feature]])
        else:
            self.target_encoder.classes_ = numpy.array(stats_dict['target_classes'],dtype='object')

        
        self._y = torch.tensor(self.target_encoder.transform(self._df_ml[[target_feature]]),
                               dtype=torch.float32)

        self.stats_dict = {
            'input_mean': [float(v1) for v1 in self.input_scaler.mean_],
            'input_scale': [float(v1) for v1 in self.input_scaler.scale_],
            'target_classes': list(self.target_encoder.classes_),
        }
        
    def _repr_html_(self):
        return f'''
        <h1>Climate Zones Dataset</h1>
        Number of samples {len(self._X)}
        '''
    
    def __len__(self):
        return len(self._X)

    def __getitem__(self,idx):
        return self._X[idx], self._y[idx]


        

We now intialise the validate and test set data loaders. Note that we initialise the preprocessing objects with the values learned from the training data, rather than calculating them on the validate or test data.

### Using our dataset class
Once we have defined the class, we can now initialise objects from it. A typical pattern is to define sperate objects for the train, validate and test sets. One then defines an iterator, i.e. a data loader object, for each of the dataset objects. Data loaders are a generic class defined by pytorch that should work with any dataset that complies with the standard interface.

In [28]:
cz_train_ds = ClimateZonesDataset(train_df, predictors, target_var, device)
cz_train_ds

In [29]:
cz_val_ds = ClimateZonesDataset(val_df, predictors, target_var, device, cz_train_ds.stats_dict)
cz_test_ds = ClimateZonesDataset(test_df, predictors, target_var, device, cz_train_ds.stats_dict)

/gws/nopw/j04/mohc_shared/users/shaddad/venv/ai4c_nb_gpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/gws/nopw/j04/mohc_shared/users/shaddad/venv/ai4c_nb_gpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [30]:
cz_val_ds[10:13]

(tensor([[-0.4408, -0.3281, -0.1737, -0.2187, -0.2642, -0.3904, -0.3665, -0.3730,
          -0.4555, -0.4177, -0.4437, -0.4373, -0.5886, -0.7146, -0.8674, -0.9889,
          -1.0984, -1.1859, -1.2289, -1.2352, -1.1710, -1.0693, -0.9048, -0.6668],
         [-0.5651, -0.5227, -0.5370, -0.5412, -0.5645, -0.5760, -0.5667, -0.5496,
          -0.6736, -0.6741, -0.6055, -0.5566, -0.9222, -1.0107, -1.1763, -1.3347,
          -1.4611, -1.4826, -1.4871, -1.4774, -1.4484, -1.4140, -1.2316, -0.9931],
         [-0.5389, -0.5087, -0.4206, -0.4052, -0.3257, -0.5483, -0.5183, -0.5140,
          -0.5723, -0.5571, -0.6006, -0.5566, -0.2212, -0.4738, -0.7350, -0.8927,
          -1.0188, -1.0741, -1.1504, -1.2124, -1.1016, -0.9153, -0.5810, -0.2653]]),
 tensor([[0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 1.]]))

In [31]:
cz_train_loader = torch.utils.data.DataLoader(
        cz_train_ds, batch_size=training_params['batch_size'], shuffle=True, num_workers=1,
    )
cz_val_loader = torch.utils.data.DataLoader(
        cz_val_ds, batch_size=training_params['batch_size'], shuffle=False, num_workers=1,
    )

In [32]:
count = 0
for i1 in cz_train_loader:
    print(i1)
    count +=1
    if count > 5:
        break

[tensor([[-0.6166, -0.6325, -0.6631, -0.6054, -0.2036,  0.3014,  0.9760,  1.4049,
          0.7887,  0.0072, -0.6899, -0.6414,  1.5725,  1.6019,  1.5301,  1.4253,
          1.2566,  1.0305,  0.8985,  0.8783,  1.0183,  1.2681,  1.5617,  1.6201],
        [-0.5203, -0.5124, -0.4935, -0.4917, -0.5952, -0.5833, -0.5624, -0.5233,
         -0.6157, -0.6555, -0.6301, -0.5513, -0.9069, -1.0744, -1.2645, -1.4492,
         -1.5739, -1.5729, -1.5552, -1.5146, -1.4636, -1.4385, -1.2547, -0.9585],
        [-0.5684, -0.5720, -0.5885, -0.6083, -0.6307, -0.6370, -0.6650, -0.7107,
         -0.6953, -0.6770, -0.6458, -0.6131, -0.8794, -1.1989, -1.4974, -1.6416,
         -1.7110, -1.7707, -1.7845, -1.7754, -1.7541, -1.6218, -1.3154, -0.9460],
        [ 3.3721,  3.6299,  2.6508,  0.2427, -0.6335, -0.7005, -0.7207, -0.7750,
         -0.7578, -0.4587,  1.3616,  3.1576,  1.2695,  1.2062,  1.0766,  0.9306,
          0.7524,  0.5662,  0.5144,  0.6071,  0.8037,  1.0505,  1.2176,  1.2625],
        [-0.2651, -0.33

## Building a pytorch model
The next step is build a class to encapsulate the architecture of the ML model that we are going to train. As with the dataset, we do this by


### Further reading
- [Classification tutorial - Pytorch docs](https://docs.pytorch.org/tutorials/beginner/blitz/cifar10_tutorial.html)
- [Classification tutorial - Machine Learning Mastery](https://machinelearningmastery.com/building-a-multiclass-classification-model-in-pytorch/ )

In [33]:
class ClimateZoneClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(24, 60)
        self.act1 = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(60, 60)
        self.act2 = torch.nn.ReLU()
        self.layer3 = torch.nn.Linear(60, 60)
        self.act3 = torch.nn.ReLU()
        self.output = torch.nn.Linear(60, 5)
        self.sigmoid = torch.nn.Sigmoid()
        self.lsm = torch.nn.Softmax(dim=-1)
 
    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        # x = self.sigmoid(self.output(x))
        x = self.lsm(self.output(x))
        return x
    


In [34]:
cz_classifier = ClimateZoneClassifier().to(device)

In [35]:
cz_classifier(cz_train_ds[:10][0].to(device))

tensor([[0.2265, 0.2168, 0.1832, 0.1801, 0.1935],
        [0.2267, 0.2167, 0.1832, 0.1802, 0.1931],
        [0.2268, 0.2166, 0.1834, 0.1802, 0.1930],
        [0.2269, 0.2166, 0.1834, 0.1802, 0.1929],
        [0.2271, 0.2165, 0.1834, 0.1802, 0.1928],
        [0.2267, 0.2163, 0.1838, 0.1801, 0.1931],
        [0.2260, 0.2162, 0.1843, 0.1799, 0.1935],
        [0.2251, 0.2161, 0.1848, 0.1799, 0.1941],
        [0.2251, 0.2160, 0.1849, 0.1798, 0.1941],
        [0.2246, 0.2159, 0.1853, 0.1797, 0.1945]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)

In [36]:
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cz_classifier.parameters(), 
                             lr=training_params['learning_rate'])

### Experiment tracking with mlflow
In a project to develop a machine learning model, we are likely to train many different models as part of our epxerimentation, with different prdictors, hyperparameters, architectures and other experimental choices that we vary to understand the problem and find the best solution. We will ave a collection of different models, and to properly asses our experiments we need to know exactly which set of choices go with which model. Experiment tracking tools log all the elements of a training run together so they can be retrieved and analysed later. A common tool for this is **ML Flow**.

Further Reading
- [ML Flow docs](https://mlflow.org/)
- [Tracking pytorch with mlflow](https://mlflow.org/docs/latest/ml/deep-learning/pytorch/)


In [37]:
import mlflow

In [38]:
try:
    mlflow_port = os.environ['MLFLOW_PORT']
except KeyError:
    mlflow_port = 4455    
mlflow_server_uri = f'http://localhost:{mlflow_port}'


In [39]:
print(f'connecting to mlflow server {mlflow_server_uri}')
mlflow.set_tracking_uri(mlflow_server_uri)

connecting to mlflow server http://localhost:4455


In [40]:
mlflow.pytorch.autolog()

In [41]:
exp_name='climate_zones_torch_nn'

In [44]:
if mlflow.get_experiment_by_name(exp_name) is None:
    exp_id = mlflow.create_experiment(exp_name)
exp1 = mlflow.get_experiment_by_name(exp_name)
exp1

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1772795630343, experiment_id='2', last_update_time=1772795630343, lifecycle_stage='active', name='climate_zones_torch_nn', tags={}>

In [45]:
cz_signature = mlflow.models.infer_signature(cz_train_ds[:5][0].numpy(), cz_train_ds[:5][1].numpy())

## Run the training loop

In pytorch, we will explcitly define a loop which cycles through the data and updates the weights of the network to produce better predictions, which is to say predictions that are closer to the target data provided. Key steps in the training loop are:

- Iterate through the full data set for the number of epochs specified. An epopch is one full pass through the data.
- In each epoch, divide the dataset into minibatches. This is a set of data for which one will do graident descent.
- For each minibatch, do the follwing steps:
  -  Do a forward pass, or inference step to make predictions with the current weights. At the start of the process these will essentially be nonsense, but as we refine the weights the performance improves.
  -  Calculate the loss or error of the predictions.
  -  Do back propogation to update the weights towards weights that would produce correct predictions. This is done by calculating the gradient in the weights and moving the weights The amount one updates the weight is based on the learning rate.
  -  Update the weights based on the calculated gradients.

### Further reading
- [Training a model - pytorch docs](https://docs.pytorch.org/tutorials/beginner/introyt/trainingyt.html)
- [ Introduction to NNs - Kaggle](https://www.kaggle.com/code/carlosaguayo/introduction-to-neural-networks/notebook)
- [Tutorial on NNs for weather](https://github.com/MetOffice/ml_weather_tutorial/blob/main/03_algorithm_selection.ipynb)

In [67]:
def get_classification_metrics(model, cz_data, set_label, target_encoder):
    class_labels = target_encoder.classes_
    return pandas.DataFrame({
        'climate_group': class_labels,
        f'precision_{set_label}': sklearn.metrics.precision_score(
            target_encoder.inverse_transform(model(cz_data._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(cz_data._y.numpy()),
            average=None,
            labels=class_labels,
        ),
        f'recall_{set_label}': sklearn.metrics.recall_score(
            target_encoder.inverse_transform(model(cz_data._X.to(device)).to('cpu').detach().numpy()),
            target_encoder.inverse_transform(cz_data._y.numpy()),
            average=None,
            labels=class_labels,
        ),
    })

In [47]:
num_epochs = training_params['num_epochs']
# num_epochs = 1 # for debug

In [48]:
%%time
with mlflow.start_run(experiment_id=exp1.experiment_id) as current_run:
    print(current_run.info.run_id)
    mlflow.log_params(training_params)    
    mlflow.log_dict(cz_train_ds.stats_dict, 'stats.json')
    for epoch in range(num_epochs):
        print(f'epoch {epoch}')
        cz_classifier.train()
        epoch_loss_train = 0.0
        for batch_X, batch_y in cz_train_loader:
            optimizer.zero_grad()
            predictions = cz_classifier(batch_X.to(device))
            loss = loss_fn(predictions, batch_y.to(device))
            loss.backward()
            optimizer.step()
            epoch_loss_train += loss.to('cpu').item()

        #divide by number of batches
        epoch_loss_train /= len(cz_train_loader)
        
        epoch_loss_val = 0.0
        for X_val, y_val in cz_val_loader:
            epoch_loss_val += loss_fn(cz_classifier(X_val.to(device)), y_val.to(device)).item()    
        epoch_loss_val /= len(cz_val_loader)

        mlflow.log_metrics(
            { 'cross_entropy_train': epoch_loss_train,
            'cross_entropy_val': epoch_loss_val, },
            step=epoch,
        )
            
    metrics_df = get_classification_metrics(cz_classifier,
                                               cz_train_ds, 
                                               'train', 
                                               target_encoder=cz_train_ds.target_encoder)
    
    metrics_df = metrics_df.merge( get_classification_metrics(cz_classifier,
                                               cz_val_ds, 
                                               'val', 
                                               target_encoder=cz_train_ds.target_encoder), on='climate_group')
    mlflow.log_table(metrics_df, 'metrics.json')
        
    torch.save(cz_classifier,'cz_model.pth')
    mlflow.log_artifact('cz_model.pth')
    
    # mlflow.pytorch.log_model(cz_classifier,
    #                      name='climate_zones_classifier_torch', 
    #                      # signature=cz_signature, 
    #                      # input_example = cz_train_ds[:5][0].numpy(),
    #                      # export_model=True,
    #                     )


c8474e1b1dc849cbb9736e30f5e12501
epoch 0
epoch 1
epoch 2
epoch 3
epoch 4
epoch 5
epoch 6
epoch 7
epoch 8
epoch 9
🏃 View run upset-mouse-812 at: http://localhost:4455/#/experiments/2/runs/c8474e1b1dc849cbb9736e30f5e12501
🧪 View experiment at: http://localhost:4455/#/experiments/2
CPU times: user 6min 51s, sys: 1min, total: 7min 51s
Wall time: 12min 6s


### Load model and do inference

In [49]:
mlflow.search_runs(exp1.experiment_id)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.cross_entropy_val,metrics.cross_entropy_train,params.criterion,params.batch_size,params.learning_rate,params.loss,params.optimizer,params.num_epochs,tags.mlflow.loggedArtifacts,tags.mlflow.source.name,tags.mlflow.source.type,tags.mlflow.user,tags.mlflow.runName
0,c8474e1b1dc849cbb9736e30f5e12501,2,FINISHED,mlflow-artifacts:/2/c8474e1b1dc849cbb9736e30f5...,2026-03-11 12:13:26.866000+00:00,2026-03-11 12:25:31.838000+00:00,0.942017,0.933169,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,"[{""path"": ""metrics.json"", ""type"": ""table""}]",/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,upset-mouse-812
1,58c2e476c4ac4fb596d4d7d5ec512ab9,2,FINISHED,mlflow-artifacts:/2/58c2e476c4ac4fb596d4d7d5ec...,2026-03-10 16:31:47.987000+00:00,2026-03-10 16:39:41.263000+00:00,0.938988,0.935487,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,"[{""path"": ""metrics.json"", ""type"": ""table""}]",/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,beautiful-mule-483
2,b14c877d8b554923be684b9ea4f8f208,2,FINISHED,mlflow-artifacts:/2/b14c877d8b554923be684b9ea4...,2026-03-10 15:57:49.618000+00:00,2026-03-10 15:58:40.695000+00:00,0.939722,0.953279,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,"[{""path"": ""metrics.json"", ""type"": ""table""}]",/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,gaudy-horse-269
3,516bda90a6f4427ab3d00ef436f59ad9,2,FINISHED,mlflow-artifacts:/2/516bda90a6f4427ab3d00ef436...,2026-03-10 15:51:21.018000+00:00,2026-03-10 15:51:30.622000+00:00,1.607479,0.000079,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,"[{""path"": ""metrics.json"", ""type"": ""table""}]",/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,brawny-goose-26
4,966179ca949c4063b2586b994cf20972,2,FINISHED,mlflow-artifacts:/2/966179ca949c4063b2586b994c...,2026-03-10 15:28:24.955000+00:00,2026-03-10 15:28:28.743000+00:00,1.602723,0.000079,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,None,/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,powerful-vole-487
5,27afb603647c438c80b0b9451953ac2c,2,FAILED,mlflow-artifacts:/2/27afb603647c438c80b0b94519...,2026-03-10 15:28:02.614000+00:00,2026-03-10 15:28:06.283000+00:00,1.599319,0.000079,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,None,/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,merciful-mole-906
6,c6bbb1a41eec4e0db3ae39847462f8f4,2,FINISHED,mlflow-artifacts:/2/c6bbb1a41eec4e0db3ae398474...,2026-03-10 14:58:22.192000+00:00,2026-03-10 14:59:10.661000+00:00,1.612314,0.000079,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,None,/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,stately-finch-140
7,cc87b94f565d4e988c77708d351d7a8c,2,FAILED,mlflow-artifacts:/2/cc87b94f565d4e988c77708d35...,2026-03-10 14:55:08.911000+00:00,2026-03-10 14:55:10.633000+00:00,NaN,NaN,None,None,None,None,None,None,None,/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,chill-shoat-369
8,7162b267d38b4024828044a25298e156,2,FINISHED,mlflow-artifacts:/2/7162b267d38b4024828044a252...,2026-03-10 14:49:43.795000+00:00,2026-03-10 14:51:10.898000+00:00,1.606192,0.000079,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,"[{""path"": ""metrics.json"", ""type"": ""table""}]",/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,unruly-fowl-893
9,74b0917762174e46a662303343fa9c50,2,FAILED,mlflow-artifacts:/2/74b0917762174e46a662303343...,2026-03-10 14:49:11.318000+00:00,2026-03-10 14:49:16.712000+00:00,1.606165,0.000079,CrossEntropyLoss,16,0.001,CrossEntropyLoss,Adam,10,None,/gws/nopw/j04/mohc_shared/users/shaddad/venv/a...,NOTEBOOK,shaddad,whimsical-hen-702


In [50]:
model_inference = torch.load('cz_model.pth', weights_only=False)


In [51]:
model_inference

ClimateZoneClassifier(
  (layer1): Linear(in_features=24, out_features=60, bias=True)
  (act1): ReLU()
  (layer2): Linear(in_features=60, out_features=60, bias=True)
  (act2): ReLU()
  (layer3): Linear(in_features=60, out_features=60, bias=True)
  (act3): ReLU()
  (output): Linear(in_features=60, out_features=5, bias=True)
  (sigmoid): Sigmoid()
  (lsm): Softmax(dim=-1)
)

In [55]:
cz_train_ds.target_encoder.inverse_transform(
    model_inference(cz_val_ds._X.to(device)).to('cpu').detach().numpy()
)
        

array(['A', 'B', 'E', ..., 'E', 'C', 'D'], shape=(40644,), dtype='<U1')

### Evaluation

We will go into more details on evaluating the performance of a machine learning model in a separate notebook.

In [ ]:
# load metrics from mlflow to see training

In [58]:
metrics_path = pathlib.Path(mlflow.artifacts.download_artifacts('runs:/c8474e1b1dc849cbb9736e30f5e12501/metrics.json'))
metrics_path

PosixPath('/tmp/tmpmnydw_35/metrics.json')

In [59]:
with open(metrics_path) as metrics_file:
    metrics_mlflow_dict = json.load(metrics_file)
metrics_mlflow_dict    

{'columns': ['climate_group',
  'precision_train',
  'recall_train',
  'precision_val',
  'recall_val'],
 'data': [['A', 0.9891433966, 0.945634619, 0.9885804292, 0.9434423149],
  ['B', 0.9360421122, 0.986016046, 0.9343065693, 0.9875998898],
  ['C', 0.8379201526, 0.9585274735, 0.8264364394, 0.9519890261],
  ['D', 0.9730827328, 0.939062276, 0.9724493769, 0.9355624239],
  ['E', 0.9943433265, 0.9797332836, 0.9931987099, 0.9788542602]]}

In [62]:
pandas.DataFrame(metrics_mlflow_dict['data'],columns=metrics_mlflow_dict['columns'])

,climate_group,precision_train,recall_train,precision_val,recall_val
0,A,0.989143,0.945635,0.988580,0.943442
1,B,0.936042,0.986016,0.934307,0.987600
2,C,0.837920,0.958527,0.826436,0.951989
3,D,0.973083,0.939062,0.972449,0.935562
4,E,0.994343,0.979733,0.993199,0.978854


In [ ]:
cal

In [ ]:
# calculate metrics from predictions on test vset

In [71]:
test_metrics_df = get_classification_metrics(cz_classifier,
                                             cz_test_ds, 
                                             'test', 
                                             target_encoder=cz_train_ds.target_encoder)
test_metrics_df


,climate_group,precision_test,recall_test
0,A,0.990693,0.948436
1,B,0.934833,0.986761
2,C,0.843255,0.952365
3,D,0.971295,0.939277
4,E,0.993372,0.978310


# Exercises

Once you have worked through the tutorial above, you can test your knowledge by adapting the code in the tutorial to experiment with different options for training the model and comparing results.

### Varying model architecture
Try to change the model architecture e.g. more layers

In [ ]:
# insert code here

### Next steps or potential follow on material

Links from notebook
- [pytorch docs](https://pytorch.org/)

Additional excercises in this tutorial material includes:
- Training a CNN autoencoder on CMIP6 data
- Training a RNN on Jena weather dataset for time series prediction. 


### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)
